In [2]:
from reedsolo import RSCodec, rs_calc_syndromes
import os
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt

import sys
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

from rs.channels import qsc_erasure_channel
from rs.dataset_gen import bytes_to_bits, get_zero_mask, RSPositionDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

from torchvision.ops import sigmoid_focal_loss
from rs.decoder import HybridDecoder


Device: cpu


In [3]:
class PositionPredictor(nn.Module):
    def __init__(self, use_batchnorm=False, dropout_rate=0.0):
        super().__init__()

        layers = []

        layers.append(nn.Linear(511, 512))
        if use_batchnorm:
            layers.append(nn.BatchNorm1d(512))
        layers.append(nn.ReLU())
        if dropout_rate > 0:
            layers.append(nn.Dropout(dropout_rate))
        
        for _ in range(3):
            layers.append(nn.Linear(512, 512))
            if use_batchnorm:
                layers.append(nn.BatchNorm1d(512))
            layers.append(nn.ReLU())
            if dropout_rate > 0:
                layers.append(nn.Dropout(dropout_rate))
        
        layers.append(nn.Linear(512, 255))

        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

In [4]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=1.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    
    def forward(self, inputs, targets):
        return sigmoid_focal_loss(inputs, targets, alpha=self.alpha, gamma=self.gamma, reduction='mean')
    
def get_criterion(loss_type):
    if loss_type == 'bce':
        return nn.BCEWithLogitsLoss()
    if loss_type == 'bce_weight':
        return nn.BCEWithLogitsLoss(pos_weight=torch.tensor([3.0]).to(device))
    elif loss_type == 'focal':
        return FocalLoss()

In [5]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, p_err, p_erase, num_samples=1000):
    model.eval()
    rsc = RSCodec(32)
    hybrid = HybridDecoder(model)

    classic_success = 0
    hybrid_success = 0
    classic_hint_success = 0

    for _ in range(num_samples):
        msg = os.urandom(223)
        codeword = rsc.encode(msg)
        noisy, erasure_pos = qsc_erasure_channel(codeword, p_err, p_erase)

        try:
            decoded, _, _ = rsc.decode(noisy)
            if bytes(decoded) == msg:
                classic_success += 1
        except:
            pass

        try:
            decoded, _, _ = rsc.decode(noisy, erase_pos=erasure_pos)
            if bytes(decoded) == msg:
                classic_hint_success += 1
        except:
            pass

        decoded = hybrid.decode(noisy, device)
        if decoded == msg:
            hybrid_success += 1
    
    return {
        'classic': classic_success / num_samples,
        'hybrid': hybrid_success / num_samples,
        'classic_hint': classic_hint_success / num_samples
    }

In [6]:
def run_experiment(config, dataset, epochs=30):
    loader = DataLoader(dataset, batch_size=256, shuffle=True)

    model = PositionPredictor(
        use_batchnorm=config.get('batchnorm', False),
        dropout_rate=config.get('dropout', 0.0)
    ).to(device)

    criterion = get_criterion(config.get('loss', 'bce'))
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    for epoch in range(epochs):
        loss = train_epoch(model, loader, criterion, optimizer)
    
    result = evaluate(model, P_ERR, P_ERASE_TEST)
    result['config'] = config
    result['final_loss'] = loss

    return result, model

In [7]:
P_ERR = 0.02
P_ERASE_TRAIN = 0.06
P_ERASE_TEST = 0.06
TRAIN_SIZE = 50000

print("Generating dataset...")
dataset = RSPositionDataset(TRAIN_SIZE, P_ERR, P_ERASE_TRAIN)

Generating dataset...


In [10]:
print("=== Comparing Loss functions ===")
print(f'{"Loss":<15} {"Hybrid":<10} {"Classic":<10}')
print("-" * 35)

loss_results = []
for loss_type in ['bce', 'bce_weight', 'focal']:
    config = {'loss' : loss_type}
    result, _ = run_experiment(config, dataset)
    loss_results.append(result)
    print(f'{loss_type:<15} {result["hybrid"]:<10.3f} {result["classic"]:<10.3f}')

=== Comparing Loss functions ===
Loss            Hybrid     Classic   
-----------------------------------
bce             0.618      0.183     
bce_weight      0.182      0.186     
focal           0.694      0.192     


In [11]:
best_loss = max(loss_results, key=lambda x: x['hybrid'])['config']['loss']
print(f'Best loss: {best_loss}')

Best loss: focal


In [12]:
print("=== Tuning BCE weight ===")
for w in [1.5, 2.0, 2.5, 3.0]:
    model = PositionPredictor().to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([w]).to(device))
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loader = DataLoader(dataset, batch_size=256, shuffle=True)
    
    for _ in range(30):
        train_epoch(model, loader, criterion, optimizer)
    
    result = evaluate(model, P_ERR, P_ERASE_TEST)
    
    model.eval()
    x = torch.tensor(dataset[0][0]).unsqueeze(0).to(device)
    with torch.no_grad():
        preds = (torch.sigmoid(model(x)) > 0.3).sum().item()
    
    print(f"weight={w:<4} Hybrid={result['hybrid']:.3f} Pred={preds}")

=== Tuning BCE weight ===
weight=1.5  Hybrid=0.527 Pred=28
weight=2.0  Hybrid=0.391 Pred=28
weight=2.5  Hybrid=0.172 Pred=35
weight=3.0  Hybrid=0.226 Pred=33


In [13]:
print("=== Tuning BCE weight (fine) ===")
for w in [1.0, 1.1, 1.2, 1.3, 1.4, 1.5]:
    model = PositionPredictor().to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([w]).to(device))
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loader = DataLoader(dataset, batch_size=256, shuffle=True)
    
    for _ in range(30):
        train_epoch(model, loader, criterion, optimizer)
    
    result = evaluate(model, P_ERR, P_ERASE_TEST)
    
    model.eval()
    x = torch.tensor(dataset[0][0]).unsqueeze(0).to(device)
    with torch.no_grad():
        preds = (torch.sigmoid(model(x)) > 0.3).sum().item()
    
    print(f"weight={w:<4} Hybrid={result['hybrid']:.3f} Pred={preds}")

=== Tuning BCE weight (fine) ===
weight=1.0  Hybrid=0.647 Pred=23
weight=1.1  Hybrid=0.578 Pred=25
weight=1.2  Hybrid=0.616 Pred=20
weight=1.3  Hybrid=0.529 Pred=28
weight=1.4  Hybrid=0.537 Pred=25
weight=1.5  Hybrid=0.481 Pred=27


In [14]:
print("\n=== Tuning Focal Loss ===")
for alpha in [0.1, 0.15, 0.2, 0.25]:
    for gamma in [1.0, 2.0]:
        model = PositionPredictor().to(device)
        criterion = FocalLoss(alpha=alpha, gamma=gamma)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
        loader = DataLoader(dataset, batch_size=256, shuffle=True)
        
        for _ in range(30):
            train_epoch(model, loader, criterion, optimizer)
        
        result = evaluate(model, P_ERR, P_ERASE_TEST)
        
        model.eval()
        x = torch.tensor(dataset[0][0]).unsqueeze(0).to(device)
        with torch.no_grad():
            preds = (torch.sigmoid(model(x)) > 0.3).sum().item()
        
        print(f"α={alpha:<4} γ={gamma:<3} Hybrid={result['hybrid']:.3f} Pred={preds}")


=== Tuning Focal Loss ===
α=0.1  γ=1.0 Hybrid=0.597 Pred=12
α=0.1  γ=2.0 Hybrid=0.666 Pred=20
α=0.15 γ=1.0 Hybrid=0.659 Pred=16
α=0.15 γ=2.0 Hybrid=0.637 Pred=18
α=0.2  γ=1.0 Hybrid=0.686 Pred=18
α=0.2  γ=2.0 Hybrid=0.638 Pred=24
α=0.25 γ=1.0 Hybrid=0.735 Pred=19
α=0.25 γ=2.0 Hybrid=0.597 Pred=23


In [27]:
print('=== Dropout tuning (BN = ON) ===')
print(f'{"Dropout":<15} {"Hybrid":<10} {"Classic":<10}')
print('-' * 35)

dropout_results_bn = []
for dropout in [0.0, 0.1, 0.3]:
    config = {'loss': best_loss, 'batchnorm': True, 'dropout': dropout}
    result, model = run_experiment(config, dataset)
    dropout_results_bn.append(result)
    print(f'{dropout:<15} {result["hybrid"]:<10.3f} {result["classic"]:<10.3f}')

=== Dropout tuning (BN = ON) ===
Dropout         Hybrid     Classic   
-----------------------------------
0.0             0.638      0.172     
0.1             0.810      0.190     
0.3             0.493      0.196     


In [28]:
print('=== Dropout tuning (BN = OFF) ===')
print(f'{"Dropout":<15} {"Hybrid":<10} {"Classic":<10}')
print('-' * 35)

dropout_results = []
for dropout in [0.0, 0.1, 0.3]:
    config = {'loss': best_loss, 'batchnorm': False, 'dropout': dropout}
    result, model = run_experiment(config, dataset)
    dropout_results.append(result)
    print(f'{dropout:<15} {result["hybrid"]:<10.3f} {result["classic"]:<10.3f}')

=== Dropout tuning (BN = OFF) ===
Dropout         Hybrid     Classic   
-----------------------------------
0.0             0.707      0.215     
0.1             0.460      0.209     
0.3             0.243      0.197     


In [1]:
best_dropout = max(dropout_results, key=lambda x: x['hybrid'])['config']['dropout']
print(best_dropout)

NameError: name 'dropout_results' is not defined

In [8]:
print("=== Grid Search ===")
print(f"{'Loss':<10} {'BN':<8} {'Drop':<8} {'Hybrid':<10} {'Pred':<8}")
print("-" * 50)

configs = [
    {'loss': 'bce', 'batchnorm': False, 'dropout': 0.0},
    {'loss': 'bce', 'batchnorm': True, 'dropout': 0.0},
    {'loss': 'bce', 'batchnorm': True, 'dropout': 0.1},
    {'loss': 'focal', 'batchnorm': False, 'dropout': 0.0},
    {'loss': 'focal', 'batchnorm': True, 'dropout': 0.0},
    {'loss': 'focal', 'batchnorm': True, 'dropout': 0.1},
]

grid_results = []

for config in configs:
    result, model = run_experiment(config, dataset)
    
    model.eval()
    x = torch.tensor(dataset[0][0]).unsqueeze(0).to(device)
    with torch.no_grad():
        preds = (torch.sigmoid(model(x)) > 0.3).sum().item()
    
    result['preds'] = preds
    grid_results.append(result)
    
    print(f"{config['loss']:<10} {str(config['batchnorm']):<8} {config['dropout']:<8} {result['hybrid']:<10.3f} {preds:<8}")

best = max(grid_results, key=lambda x: x['hybrid'])
print(f"\nBest: {best['config']}")
print(f"Hybrid FSR: {best['hybrid']:.3f}")

=== Grid Search ===
Loss       BN       Drop     Hybrid     Pred    
--------------------------------------------------
bce        False    0.0      0.586      19      
bce        True     0.0      0.552      23      
bce        True     0.1      0.778      19      
focal      False    0.0      0.685      17      
focal      True     0.0      0.633      18      
focal      True     0.1      0.810      18      

Best: {'loss': 'focal', 'batchnorm': True, 'dropout': 0.1}
Hybrid FSR: 0.810


In [9]:
print("=== Tuning Focal Loss (BN=True, Dropout=0.1) ===")
print(f"{'alpha':<8} {'gamma':<8} {'Hybrid':<10} {'Pred':<8}")
print("-" * 40)

focal_results = []

for alpha in [0.15, 0.2, 0.25, 0.3]:
    for gamma in [1.0, 1.5, 2.0]:
        model = PositionPredictor(use_batchnorm=True, dropout_rate=0.1).to(device)
        criterion = FocalLoss(alpha=alpha, gamma=gamma)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
        loader = DataLoader(dataset, batch_size=256, shuffle=True)
        
        for _ in range(30):
            train_epoch(model, loader, criterion, optimizer)
        
        result = evaluate(model, P_ERR, P_ERASE_TEST)
        
        model.eval()
        x = torch.tensor(dataset[0][0]).unsqueeze(0).to(device)
        with torch.no_grad():
            preds = (torch.sigmoid(model(x)) > 0.3).sum().item()
        
        focal_results.append({'alpha': alpha, 'gamma': gamma, 'hybrid': result['hybrid'], 'preds': preds})
        print(f"{alpha:<8} {gamma:<8} {result['hybrid']:<10.3f} {preds:<8}")

best_focal = max(focal_results, key=lambda x: x['hybrid'])
print(f"\nBest Focal: α={best_focal['alpha']}, γ={best_focal['gamma']}")
print(f"Hybrid FSR: {best_focal['hybrid']:.3f}")

=== Tuning Focal Loss (BN=True, Dropout=0.1) ===
alpha    gamma    Hybrid     Pred    
----------------------------------------
0.15     1.0      0.778      16      
0.15     1.5      0.804      16      
0.15     2.0      0.812      18      
0.2      1.0      0.800      16      
0.2      1.5      0.825      17      
0.2      2.0      0.797      18      
0.25     1.0      0.814      17      
0.25     1.5      0.797      19      
0.25     2.0      0.726      19      
0.3      1.0      0.817      17      
0.3      1.5      0.797      18      
0.3      2.0      0.706      18      

Best Focal: α=0.2, γ=1.5
Hybrid FSR: 0.825
